# TASK 7.7 — Xây dữ liệu v2 và kiểm toán trước bàn giao

Mục tiêu hiện tại: 14.400 ảnh, gồm 3.000 replay (real + Stable Diffusion),
5.350 fake mới và 6.050 real mới. Replay fake Midjourney đã bị loại theo config.
Đây là cấu hình mục tiêu, chưa phải kết quả nghiệm thu.

Bước 4 kiểm tra cấu trúc tar trước khi hash, hỗ trợ CFE `rel_path` và nhãn benchmark
`0_real/1_fake`, `real/fake`, `nature/ai`. Không đoán nhãn hay bỏ reference để cho PASS.
Lỗi gốc và đường dẫn gây lỗi được in đầy đủ, đồng thời lưu trên Drive.

Tar GAN gồm 8 nhóm chính và 5 nhóm phụ theo báo cáo kiểm toán Drive. Index giữ toàn bộ ảnh; coverage 18 nhóm chính và 5 nhóm phụ được báo riêng.


---  
## BƯỚC 1: Mount Google Drive & Thiết Lập Thư Mục Làm Việc

In [ ]:
from google.colab import drive
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import os, sys, subprocess, json, shutil

# 1. Mount Google Drive (Dung de doc datasets va luu ban giao Phase 7)
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Fatformer')
if not DRIVE_ROOT.exists():
    for alt in ['/content/drive/MyDrive/FatFormer_Hub', '/content/drive/MyDrive/fatformer']:
        if Path(alt).exists():
            DRIVE_ROOT = Path(alt)
            break
print(f'[✓] Google Drive Root: {DRIVE_ROOT}')

# 2. Lay ma nguon truc tiep tu Git ve SSD cuc bo Colab (/content/fatformer-xla)
# TOC DO CAO, KHONG NGHEN DRIVE FUSE, KHONG XUNG DOT MERGE/AUTOSAVE
BRANCH = 'hoang'
REPO_URL = 'https://github.com/LeeVHoangtk3/fatformer-xla.git'
PROJECT_ROOT = Path('/content/fatformer-xla')

if (PROJECT_ROOT / '.git').exists():
    print(f'[*] Repo da ton tai tai {PROJECT_ROOT}, dang dong bo ma nguon moi nhat tu nhanh {BRANCH}...')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    print(f'[*] Dang clone ma nguon tu nhanh {BRANCH} ({REPO_URL}) ve SSD Colab...')
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)

# 3. Xac nhan workspace va nap duong dan tools vao sys.path
if not (PROJECT_ROOT / 'tools/build_manifest_v2.py').is_file():
    raise RuntimeError(f'Loi: Khong tim thay tools/build_manifest_v2.py tai {PROJECT_ROOT}!')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'tools') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'tools'))

# 4. Ham thuc thi logged tu tuc (Self-contained, khong phu thuoc import ngoai khi bootstrap)
def run_logged(command, root):
    root = Path(root)
    log_dir = root / 'DATASET/logs/phase7/commands'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.log')
    command = [str(x) for x in command]
    tail = deque(maxlen=40)
    print('Running:', ' '.join(command), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    with log_path.open('w', encoding='utf-8') as log:
        log.write('Command: ' + repr(command) + '\n')
        with subprocess.Popen(command, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1, env=env) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            code = process.wait()
    if code:
        raise RuntimeError(f'Command failed (exit {code}). Full log: {log_path}\n' + ''.join(tail))
    return log_path

def run(*args):
    return run_logged(args, PROJECT_ROOT)

print(f'[✓] Da thiet lap thanh cong workspace sieu toc tai: {PROJECT_ROOT}')
run('git', 'log', '-1', '--oneline')
BASE_CONFIG = 'configs/phase7_data_v2.yaml'
CONFIG = BASE_CONFIG

---  
## BƯỚC 2: Cài Đặt Thư Viện Phụ Trợ & Kiểm Tra Môi Trường

In [ ]:
run(sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'duckdb', 'pyarrow', 'pillow', 'scipy', 'pyyaml', 'requests', 'matplotlib')
import duckdb, pyarrow, PIL, scipy, numpy, yaml
print({m.__name__: m.__version__ for m in (duckdb, pyarrow, PIL, scipy, numpy)})
# Fingerprint của config/deny-list/replay thay đổi sẽ chặn resume.

## Bước 3 — Kiểm thử offline
Phải thành công trước khi chạy dữ liệu thật. Đây là kiểm thử cơ chế trên dữ liệu tổng hợp, không thay thế audit thực tế.

In [ ]:
run(sys.executable, 'tools/test_data_v2.py', '-v')
run(sys.executable, 'tools/test_index_test_sets.py', '-v')


---  
## BƯỚC 4: Tạo Reference Hash Index Của Các Tập Test & Sinh Deny-List
> **Mục đích**: Tạo cơ sở dữ liệu băm (SHA-256 + pHash 64-bit) đọc trực tiếp từ các file `.tar` trên Google Drive nhằm phục vụ bộ lọc kiểm toán cách ly.

In [ ]:
# Tải lại helper từ file để chạy lại riêng Bước 4 cũng nhận được log mới.
import importlib
import phase7_reference_io
importlib.reload(phase7_reference_io)
from phase7_reference_io import index_reference, run_logged

def run(*args):
    return run_logged(args, PROJECT_ROOT)

import tarfile
DRIVE_DATASETS = DRIVE_ROOT / 'datasets' if 'DRIVE_ROOT' in globals() else Path('/content/drive/MyDrive/Fatformer/datasets')
TAR_CFE_TEST = DRIVE_DATASETS / 'test_communityforensics.tar'
TAR_GANS_TEST = DRIVE_DATASETS / 'test_benchmark_gans.tar'
TAR_DIFF_TEST = DRIVE_DATASETS / 'test_benchmark_diffusion.tar'
TAR_STAGING = DRIVE_DATASETS / 'diffusion_staging.tar'
for path in (TAR_CFE_TEST, TAR_GANS_TEST, TAR_DIFF_TEST, TAR_STAGING):
    if not path.is_file():
        raise FileNotFoundError(path)

# Đọc đúng một manifest từ tar, không bỏ qua lỗi thiếu metadata.
REF_DIR = PROJECT_ROOT / 'DATASET/logs/phase7'
REF_DIR.mkdir(parents=True, exist_ok=True)
CFE_MANIFEST = REF_DIR / 'cfe_test_manifest.json'
with tarfile.open(TAR_CFE_TEST, 'r:*') as archive:
    candidates = [m for m in archive if m.isfile() and Path(m.name).name == 'cfe_manifest.json']
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một cfe_manifest.json; tìm thấy {len(candidates)}')
    with archive.extractfile(candidates[0]) as source:
        CFE_MANIFEST.write_bytes(source.read())

# Kiểm tra schema gốc của notebook chuẩn bị CFE trước khi hash.
manifest_data = json.loads(CFE_MANIFEST.read_text(encoding='utf-8-sig'))
manifest_rows = manifest_data if isinstance(manifest_data, list) else manifest_data.get('records', manifest_data.get('images', []))
if len(manifest_rows) != 2000:
    raise RuntimeError(f'CFE manifest cần 2.000 ảnh; hiện có {len(manifest_rows)}')
print('CFE manifest:', len(manifest_rows), 'records; keys:', sorted(manifest_rows[0]))
if not any(k in manifest_rows[0] for k in ('rel_path', 'relative_path', 'path', 'file_path', 'file_name', 'image_name')):
    raise RuntimeError('Manifest CFE thiếu trường đường dẫn ảnh')

# Sinh deny-list và kiểm tra replay trước khi hash/tải dữ liệu lớn.
run(sys.executable, 'tools/build_generator_denylist.py',
    '--cfe-manifest', CFE_MANIFEST,
    '--task51-report', 'docs/bao_cao/C/bao_cao_task_5.1_full_benchmark_18_datasets.md',
    '--out', REF_DIR / 'generator_denylist.json')
from build_manifest_v2 import Config, check_replay_generators
check_replay_generators(Config(str(PROJECT_ROOT / CONFIG)))

# Kiểm tra layout trước khi hash: tar GAN gồm 8 nhóm chính + 5 nhóm phụ.
# Giữ toàn bộ ảnh của crn/imle/san/seeingdark/whichfaceisreal trong reference;
# coverage của 18 nhóm Task 5.1 vẫn bắt buộc, không gộp hay bỏ nhóm phụ.
run(sys.executable, 'tools/index_test_sets.py', '--role', 'test18',
    '--tar', TAR_GANS_TEST, TAR_DIFF_TEST, '--inspect-only',
    '--out', REF_DIR / 'hashes_18sets.json.gz',
    '--layout-report', REF_DIR / 'test18_layout.json')

# Chỉ tái sử dụng index đã hoàn tất nếu input, code và checksum index không đổi.
index_reference(PROJECT_ROOT, 'cfe', [TAR_CFE_TEST], REF_DIR / 'hashes_cfe_test.json',
                manifest=CFE_MANIFEST, workers=4)
index_reference(PROJECT_ROOT, 'test18', [TAR_GANS_TEST, TAR_DIFF_TEST],
                REF_DIR / 'hashes_18sets.json.gz', workers=4)
print('Bước 4 hoàn tất: CFE và đủ 18 tập test có index hợp lệ.')


## Bước 5 — Preflight và dry-run
Builder kiểm tra replay với deny-list trước khi đọc metadata hoặc tải shard.
Nếu Midjourney replay bị chặn, dừng để giải quyết hợp đồng dữ liệu; không dùng --force để bỏ qua.

Số shard, dung lượng và thời gian phải lấy từ log thực tế. Không coi 14 shard/21,81 GB là cam kết cố định.
State cũ thiếu fingerprint phải chuyển sang thư mục state mới hoặc bắt đầu lại có chủ đích bằng --force.

In [ ]:
# Pin revision truoc khi doc metadata. Giu nguyen config goc trong Git.
from build_manifest_v2 import Config, check_replay_generators
import requests, re, yaml, json

# 1. Tu dong don dep context.json cu (neu co) truoc khi dry-run de khop chuan voi moi truong Colab
ctx_file = PROJECT_ROOT / 'DATASET/logs/phase7/build_state/context.json'
if ctx_file.exists():
    ctx_file.unlink()
    print('[*] Da lam moi context.json theo moi truong Colab Linux.')

# 2. Kiem tra replay va pin revision
check_replay_generators(Config(str(PROJECT_ROOT / CONFIG)))
RUNTIME_CONFIG = PROJECT_ROOT / 'configs/phase7_data_v2.runtime.yaml'
if RUNTIME_CONFIG.exists():
    runtime = yaml.safe_load(RUNTIME_CONFIG.read_text(encoding='utf-8'))
    base = yaml.safe_load((PROJECT_ROOT / BASE_CONFIG).read_text(encoding='utf-8'))
    check = json.loads(json.dumps(runtime))
    for name, spec in check['datasets'].items():
        spec['url_template'] = base['datasets'][name]['url_template']
    if check != base:
        raise RuntimeError('Config nguồn đã đổi. Tạo runtime/state mới; không tiếp tục resume cũ.')
else:
    runtime = yaml.safe_load((PROJECT_ROOT / BASE_CONFIG).read_text(encoding='utf-8'))
    for spec in runtime['datasets'].values():
        url = spec['url_template']
        match = re.match(r'https://huggingface.co/datasets/([^/]+/[^/]+)/resolve/main/', url)
        if match:
            response = requests.get(f'https://huggingface.co/api/datasets/{match.group(1)}', timeout=60)
            response.raise_for_status()
            revision = response.json()['sha']
            if not re.fullmatch(r'[0-9a-f]{40}', revision):
                raise RuntimeError('Dataset revision không hợp lệ')
            spec['url_template'] = url.replace('/resolve/main/', f'/resolve/{revision}/')
    RUNTIME_CONFIG.write_text(yaml.safe_dump(runtime, allow_unicode=True, sort_keys=False), encoding='utf-8')
CONFIG = str(RUNTIME_CONFIG)
run(sys.executable, 'tools/build_manifest_v2.py', '--config', CONFIG, '--dry-run')

---  
## BƯỚC 6: Chuẩn Bị Dữ Liệu Replay In-dist (3.600 Mẫu)
> **Mục đích**: Giải nén tập `diffusion_staging.tar` lưu trên Drive vào bộ nhớ cục bộ `/content/dataset_local/diffusion_staging`.

In [ ]:
REPLAY_DIR = Path('/content/dataset_local/diffusion_staging')
REPLAY_DIR.mkdir(parents=True, exist_ok=True)
run('tar', '-xf', TAR_STAGING, '-C', REPLAY_DIR)
print('Replay đã giải nén; builder sẽ xác minh đủ ảnh, checksum và generator.')

## Bước 7 — Build với đầy đủ reference
Ảnh nguyên byte và state nằm trong repo trên Drive; cache Parquet tạm nằm ở /content.
Resume xác minh ảnh và fingerprint trước khi bỏ qua shard. Giới hạn 4 giờ/60 GB được kiểm tra lúc tải/xử lý;
khi chạm ngưỡng, dừng và lưu checkpoint, không tự hạ 8.000 ảnh rồi công nhận đạt.

Dữ liệu mới được lọc với cả CFE và 18 tập test trước chia split. Split thử chia theo generator;
ngoại lệ theo ảnh được ghi trong metadata. Có manifest train_select để loader hiện tại không trộn val_calib.

In [ ]:
# Self-healing: Tu dong kiem tra va lam sach Shard 66 neu chua nap du 405 mau
bstate = PROJECT_ROOT / 'DATASET/logs/phase7/build_state'
cand_file = bstate / 'candidates.jsonl'
done_file = bstate / 'fetch_done.json'
if cand_file.exists() and done_file.exists():
    lines = cand_file.read_text(encoding='utf-8').splitlines()
    s66_lines = [l for l in lines if '"shard": 66' in l or '"shard":66' in l]
    if len(s66_lines) > 0 and len(s66_lines) < 400:
        print(f'[*] Phat hien Shard 66 chua nap du ({len(s66_lines)} < 405), tu dong reset cache Shard 66 de nap lai...')
        clean_lines = [l for l in lines if '"shard": 66' not in l and '"shard":66' not in l]
        cand_file.write_text('\n'.join(clean_lines) + '\n', encoding='utf-8')
        done_data = json.loads(done_file.read_text(encoding='utf-8'))
        done_data.pop('cfe_small:66', None)
        done_file.write_text(json.dumps(done_data, indent=2), encoding='utf-8')

run(sys.executable, 'tools/build_manifest_v2.py', '--config', CONFIG,
    '--cache_dir', '/content/cache_v2', '--replay_root', REPLAY_DIR,
    '--replay_copy_mode', 'copy',
    '--ref_index', REF_DIR / 'hashes_cfe_test.json', REF_DIR / 'hashes_18sets.json.gz',
    '--resume', '--strict')

---  
## BƯỚC 8: Kiểm Toán Độc Lập Chốt Chặn G7.2 (`tools/audit_manifest_v2.py`)
> **Ràng buộc bất di bất dịch**: Báo cáo kiểm toán bắt buộc phải có **`violations = 0`** và **`failures = 0`**. Nếu có bất kỳ vi phạm nào, Task 7.11 sẽ bị chặn ngay lập tức.

In [ ]:
run(sys.executable, 'tools/audit_manifest_v2.py', '--config', CONFIG,
    '--manifest', 'DATASET/manifests/data_manifest_v2.json',
    '--ref', f'cfe={REF_DIR / "hashes_cfe_test.json"}',
    '--ref', f'test18={REF_DIR / "hashes_18sets.json.gz"}',
    '--threshold_external', '8', '--threshold_internal', '3',
    '--out', REF_DIR / 'data_isolation_report.json')

from package_manifest_v2 import validate_handoff
validate_handoff('DATASET/manifests/data_manifest_v2.json',
                 REF_DIR / 'data_isolation_report.json', CONFIG)
print('Audit đầy đủ và khớp manifest/config hiện tại.')

---  
## BƯỚC 9: Đóng Gói `train_crossdomain_v2.tar` & Đẩy Lên Google Drive
> **Mục đích**: Đóng gói toàn bộ byte ảnh gốc vào kho `.tar` không nén lại (để đọc tuần tự tốc độ cao) và lưu trực tiếp lên Google Drive kèm chữ ký băm SHA-256.

In [ ]:
DEST_DRIVE_PHASE7 = DRIVE_ROOT / 'phase7' if 'DRIVE_ROOT' in globals() else Path('/content/drive/MyDrive/Fatformer/phase7')
DEST_DRIVE_PHASE7.mkdir(parents=True, exist_ok=True)
TAR_OUT = DEST_DRIVE_PHASE7 / 'train_crossdomain_v2.tar'

# Chi dong goi record da chon va co bao cao audit khop checksum.
# Tar giu duong dan DATASET/... de extract -C <repo>.
run(sys.executable, 'tools/package_manifest_v2.py', '--config', CONFIG,
    '--manifest', 'DATASET/manifests/data_manifest_v2.json',
    '--report', REF_DIR / 'data_isolation_report.json', '--out', TAR_OUT)

# Sao luu manifest va bao cao audit len Google Drive de C tiep nhan Task 7.11
for mf in (PROJECT_ROOT / 'DATASET/manifests').glob('data_manifest_v2*.json'):
    shutil.copy2(mf, DEST_DRIVE_PHASE7 / mf.name)
shutil.copy2(REF_DIR / 'data_isolation_report.json', DEST_DRIVE_PHASE7 / 'data_isolation_report.json')
print(f'[✓] Da tao {TAR_OUT} va sao luu toan bo manifest/audit report len Drive!')
print(f'Vui long thong bao cho Thanh vien C de xac nhan tiep nhan mo Task 7.11.')

## Bước 10 — Bàn giao có bằng chứng

Chỉ hoàn tất khi tar và checksum tồn tại, audit khớp manifest, báo cáo thực tế đã viết và C xác nhận.
Không đánh dấu hoàn thành bằng checklist cố định.

- Extract: `tar -xf train_crossdomain_v2.tar -C <repo>`.
- Train/chọn checkpoint với loader hiện tại: truyền `--train_manifest DATASET/manifests/data_manifest_v2_train_select.json`.
  File này chỉ chứa train và val_select; không đưa val_calib vào loader đang gộp mọi split khác train thành val.
- Calibration/router chỉ đọc `data_manifest_v2_val_calib.json`.
- Lưu policy chia split, số lượng sau mọi bộ lọc, thời gian/dung lượng thật và kết quả audit trong báo cáo Task 7.7.
- Nếu còn xung đột replay/generator hoặc thiếu reference: Task 7.7 và 7.11 vẫn bị chặn.